# 2. LoRA fine-tune and held-out evaluation

Fine-tunes Cosmos Policy (LoRA rank 32, 2000 iterations) on the set built by notebook 1, merges the LoRA into the base checkpoint, and evaluates it on held-out start states 25-49 with the same seeds as the base run.

**Before running:** A100 runtime, the `HF_TOKEN` secret, and notebook 1's outputs in `OUT` (`ftset_<judge>.tar`, `tasks/`, `base_heldout_r*`).

Self-contained like notebook 1: the code cells write the scripts to `/content/code`.

**Time on an A100:** setup ~15 min, training roughly 2 h, evaluation ~25 min per seed on the 5 custom tasks.

In [ ]:
OUT = '/content/drive/MyDrive/cosmos_robometer/runs'
TASKS = 'custom'
SET = 'libero'
ITERS = 2000
EVAL_SEEDS = 2

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
CODE = '/content/code'
os.makedirs(f'{CODE}/tasks', exist_ok=True)
TASK_ARGS = f'--tasks_manifest {CODE}/tasks/manifest.json' if TASKS == 'custom' else f'--suite {TASKS}'
os.environ.update(CODE=CODE, OUT=OUT, TASK_ARGS=TASK_ARGS)
os.makedirs(OUT, exist_ok=True)
os.environ.update(SET=SET, ITERS=str(ITERS), EVAL_SEEDS=str(EVAL_SEEDS))

## Code

VM setup (same as notebook 1).

In [ ]:
%%writefile /content/code/setup_colab.sh
#!/bin/bash
set -e
WANT_JUDGE=${1:-0}
export DEBIAN_FRONTEND=noninteractive
apt-get -qq update
apt-get -qq install -y libegl1 libgl1 libgl1-mesa-dri libglib2.0-0 libosmesa6 ffmpeg > /dev/null
pip -q install uv
cd /content
[ -d cosmos-policy ] || git clone -q --depth 1 https://github.com/NVlabs/cosmos-policy.git
[ -d LIBERO ] || git clone -q --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
(cd cosmos-policy && uv sync -q --extra cu128 --group libero --python 3.10) &
PIDS=$!
if [ "$WANT_JUDGE" = 1 ]; then
  [ -d robometer ] || git clone -q --depth 1 https://github.com/robometer/robometer.git
  (cd robometer && uv sync -q && uv pip install -q --python .venv/bin/python "torchao==0.13.0" h5py pyyaml pillow) &
  PIDS="$PIDS $!"
fi
hf download nvidia/Cosmos-Policy-LIBERO-Predict2-2B > /dev/null
hf download nvidia/Cosmos-Predict2-2B-Video2World tokenizer/tokenizer.pth > /dev/null
for p in $PIDS; do wait $p; done

cat > /content/env.sh <<'EOF'
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl
export __EGL_VENDOR_LIBRARY_FILENAMES=/content/10_nvidia.json
export LD_LIBRARY_PATH=/usr/lib64-nvidia:$LD_LIBRARY_PATH
export CPV=/content/cosmos-policy/.venv/bin/python
export RBV=/content/robometer/.venv/bin/python
EOF
echo '{"file_format_version": "1.0.0", "ICD": {"library_path": "/usr/lib64-nvidia/libEGL_nvidia.so.0"}}' > /content/10_nvidia.json
CPV=/content/cosmos-policy/.venv/bin/python
L=$($CPV -c "import importlib.util, os; print(os.path.dirname(importlib.util.find_spec('libero').origin))")
[ -d "$L/libero" ] && L=$L/libero
mkdir -p /root/.libero
printf "benchmark_root: %s\nbddl_files: %s/bddl_files\ninit_states: %s/init_files\ndatasets: %s/../datasets\nassets: /content/LIBERO/libero/libero/assets\n" "$L" "$L" "$L" "$L" > /root/.libero/config.yaml
[ -e $L/assets ] || ln -s /content/LIBERO/libero/libero/assets $L/assets
echo SETUP_DONE

Cosmos Policy's own trainer, with one fix: the LoRA layers start from zero in the checkpoint and are re-initialized.

In [ ]:
%%writefile /content/code/train_lora.py
import runpy

import torch

import cosmos_policy._src.imaginaire.utils.checkpoint_db as checkpoint_db
import cosmos_policy._src.predict2.checkpointer.dcp as dcp

_download = checkpoint_db.get_checkpoint_by_hf
_UNUSED = ('Cosmos-Predict2-2B-Video2World/model-480p-16fps.pt', 'Cosmos-Policy-ALOHA-Predict2-2B/Cosmos-Policy-ALOHA-Predict2-2B.pt')
checkpoint_db.get_checkpoint_by_hf = lambda uri: uri if uri.endswith(_UNUSED) else _download(uri)

_load = dcp.DistributedCheckpointer.load


def load_then_init_lora(self, model, *args, **kwargs):
    out = _load(self, model, *args, **kwargs)
    lora = [m for m in model.modules() if hasattr(m, 'reset_lora_parameters') and 'default' in getattr(m, 'lora_A', {})]
    if lora and all(float(m.lora_A['default'].weight.abs().sum()) == 0.0 for m in lora):
        with torch.no_grad():
            for m in lora:
                m.reset_lora_parameters('default', True)
    print(f'LORA modules {len(lora)}', flush=True)
    return out


dcp.DistributedCheckpointer.load = load_then_init_lora
runpy.run_module('cosmos_policy.scripts.train', run_name='__main__', alter_sys=True)

Merges the trained LoRA into the base checkpoint so the rollout script can load it, and saves the LoRA weights alone (small) to Drive.

In [ ]:
%%writefile /content/code/merge_lora.py
import argparse
import glob
import os

import torch
import torch.distributed.checkpoint as dcp
from torch.distributed.checkpoint import FileSystemReader

p = argparse.ArgumentParser()
p.add_argument('--ckpt', required=True, help='training checkpoint folder, e.g. .../checkpoints/iter_000002000')
p.add_argument('--out', required=True)
p.add_argument('--save_lora', default='', help='optional small file with only the LoRA weights')
a = p.parse_args()

reader = FileSystemReader(os.path.join(a.ckpt, 'model'))
meta = reader.read_metadata().state_dict_metadata
lora = {k: torch.empty(v.size, dtype=v.properties.dtype) for k, v in meta.items() if '.lora_' in k}
try:
    dcp.load(lora, storage_reader=reader)
except TypeError:
    dcp.load(lora, storage_reader=reader, no_dist=True)

if a.save_lora:
    torch.save(lora, a.save_lora)

snap = glob.glob(os.path.expanduser('~/.cache/huggingface/hub/models--nvidia--Cosmos-Policy-LIBERO-Predict2-2B/snapshots/*'))[0]
sd = torch.load(os.path.join(snap, 'Cosmos-Policy-LIBERO-Predict2-2B.pt'), map_location='cpu', weights_only=False)
n = 0
for k in [k for k in lora if '.lora_A.' in k]:
    w = k.split('.lora_A.')[0] + '.weight'
    sd[w] = (sd[w].float() + lora[k.replace('.lora_A.', '.lora_B.')].float() @ lora[k].float()).to(sd[w].dtype)
    n += 1
sd = {k: v for k, v in sd.items() if not k.endswith('_extra_state')}
torch.save(sd, a.out)
print('merged', n, 'LoRA layers into', a.out)

Policy rollouts (same as notebook 1).

In [ ]:
%%writefile /content/code/rollout.py
import argparse
import json
import os
import pickle
from types import SimpleNamespace

import h5py
import numpy as np

os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
os.environ['DETERMINISTIC'] = 'True'

import cosmos_policy._src.imaginaire.utils.checkpoint_db as checkpoint_db

_download = checkpoint_db.get_checkpoint_by_hf
_UNUSED = ('Cosmos-Predict2-2B-Video2World/model-480p-16fps.pt', 'Cosmos-Policy-ALOHA-Predict2-2B/Cosmos-Policy-ALOHA-Predict2-2B.pt')
checkpoint_db.get_checkpoint_by_hf = lambda uri: uri if uri.endswith(_UNUSED) else _download(uri)

from libero.libero import benchmark
from cosmos_policy.experiments.robot.cosmos_utils import get_action, get_model, init_t5_text_embeddings_cache, load_dataset_stats
from cosmos_policy.experiments.robot.libero.libero_utils import get_libero_dummy_action, get_libero_env
from cosmos_policy.experiments.robot.libero.run_libero_eval import TASK_MAX_STEPS, PolicyEvalConfig, prepare_observation
from cosmos_policy.utils.utils import jpeg_encode_image, set_seed_everywhere

p = argparse.ArgumentParser()
p.add_argument('--out', required=True)
p.add_argument('--suite', default='libero_spatial')
p.add_argument('--tasks_manifest', default='', help='tasks/manifest.json to run the custom cluttered tasks instead of a suite')
p.add_argument('--tasks', default='all')
p.add_argument('--episodes', default='25:50')
p.add_argument('--rseed', type=int, default=0)
p.add_argument('--scene_shift', type=float, default=0.10)
p.add_argument('--ckpt', default='nvidia/Cosmos-Policy-LIBERO-Predict2-2B')
p.add_argument('--save_traj', type=int, default=0)
a = p.parse_args()

HF = 'nvidia/Cosmos-Policy-LIBERO-Predict2-2B'
cfg = PolicyEvalConfig(
    config='cosmos_predict2_2b_480p_libero__inference_only', ckpt_path=a.ckpt, config_file='cosmos_policy/config/config.py',
    dataset_stats_path=f'{HF}/libero_dataset_statistics.json', t5_text_embeddings_path=f'{HF}/libero_t5_embeddings.pkl',
    use_wrist_image=True, use_proprio=True, normalize_proprio=True, unnormalize_actions=True,
    chunk_size=16, num_open_loop_steps=16, trained_with_image_aug=True, use_jpeg_compression=True, flip_images=True,
    num_denoising_steps_action=5, num_denoising_steps_future_state=1, num_denoising_steps_value=1,
    task_suite_name=a.suite, seed=195, deterministic=True)
stats = load_dataset_stats(cfg.dataset_stats_path)
init_t5_text_embeddings_cache(cfg.t5_text_embeddings_path)
model, _ = get_model(cfg)

if a.tasks_manifest:
    root = os.path.dirname(os.path.abspath(a.tasks_manifest))
    custom = json.load(open(a.tasks_manifest))
    n_tasks, max_steps = len(custom), 300
else:
    suite = benchmark.get_benchmark_dict()[a.suite]()
    n_tasks, max_steps = suite.n_tasks, TASK_MAX_STEPS[a.suite]
task_ids = range(n_tasks) if a.tasks == 'all' else [int(x) for x in a.tasks.split(',')]
e0, e1 = [int(x) for x in a.episodes.split(':')]


def load_task(tid):
    if not a.tasks_manifest:
        return suite.get_task(tid), suite.get_task_init_states(tid)
    t = custom[tid]
    bddl = os.path.join(root, t['bddl'])
    task = SimpleNamespace(language=t['language'], problem_folder=os.path.dirname(bddl), bddl_file=os.path.basename(bddl))
    return task, pickle.load(open(os.path.join(root, t['init']), 'rb'))


def shift_scene(env, rng):
    sim = env.env.sim
    m = sim.model
    free = [(j, m.jnt_bodyid[j]) for j in range(m.njnt) if m.jnt_type[j] == 0]
    free_bodies = {b for _, b in free}
    robotish = lambda name: name.startswith(('robot', 'mount', 'gripper'))
    fixtures = [b for b in range(1, m.nbody) if m.body_parentid[b] == 0 and b not in free_bodies
                and not robotish((m.body_id2name(b) or '').lower())
                and 'table' not in (m.body_id2name(b) or '').lower() and 'floor' not in (m.body_id2name(b) or '').lower()]
    touches_robot = lambda g: robotish((m.body_id2name(m.body_rootid[m.geom_bodyid[g]]) or '').lower())

    def contacts():
        sim.forward()
        cs = [sim.data.contact[k] for k in range(sim.data.ncon)]
        deep = sum(c.dist < -0.004 for c in cs)
        robot = sum(c.dist < -0.001 and (touches_robot(c.geom1) or touches_robot(c.geom2)) for c in cs)
        return deep, robot

    base_deep, _ = contacts()
    body_pos, qpos = m.body_pos.copy(), sim.data.qpos.copy()
    for _ in range(50):
        th = rng.uniform(0, 2 * np.pi)
        d = a.scene_shift * np.array([np.cos(th), np.sin(th)])
        for b in fixtures:
            m.body_pos[b, :2] = body_pos[b, :2] + d
        for j, _ in free:
            q = m.jnt_qposadr[j]
            sim.data.qpos[q:q + 2] = qpos[q:q + 2] + d
        deep, robot = contacts()
        if robot == 0 and deep <= base_deep:
            return True
    m.body_pos[:] = body_pos
    sim.data.qpos[:] = qpos
    sim.forward()
    return False


os.makedirs(os.path.join(a.out, 'traj'), exist_ok=True)
results = []
for tid in task_ids:
    task, init_states = load_task(tid)
    env, desc = get_libero_env(task, 'cosmos', resolution=256)
    for ep in range(e0, e1):
        env.reset()
        obs = env.set_init_state(init_states[ep])
        shifted = shift_scene(env, np.random.default_rng([0, tid, ep])) if a.scene_shift > 0 else False
        for _ in range(10):
            obs, _, _, _ = env.step(get_libero_dummy_action('cosmos'))
        prim, wrist, prop, acts, queue, success, t = [], [], [], [], [], False, 10
        while t < max_steps + 10:
            o = prepare_observation(obs, None, True)
            if not queue:
                set_seed_everywhere(a.rseed)
                r = get_action(cfg, model, stats, o, desc, seed=195 + a.rseed, randomize_seed=False,
                               num_denoising_steps_action=cfg.num_denoising_steps_action,
                               generate_future_state_and_value_in_parallel=True, batch_size=1)
                queue = [np.asarray(x) for x in r['actions']]
            act = queue.pop(0)
            prim.append(o['primary_image'])
            wrist.append(o['wrist_image'])
            prop.append(o['proprio'])
            acts.append(act)
            obs, _, done, _ = env.step(act.tolist())
            if done:
                success = True
                break
            t += 1
        row = dict(task=tid, ep=ep, rseed=a.rseed, success=success, steps=t, shifted=shifted)
        results.append(row)
        print(json.dumps(row), flush=True)
        with open(os.path.join(a.out, 'episodes.jsonl'), 'a') as f:
            f.write(json.dumps(row) + '\n')
        if a.save_traj:
            with h5py.File(os.path.join(a.out, 'traj', f'task{tid}_ep{ep}_r{a.rseed}_success{int(success)}.hdf5'), 'w') as f:
                jpeg = h5py.vlen_dtype(np.dtype('uint8'))
                f.create_dataset('primary_images_jpeg', data=[jpeg_encode_image(x, quality=95) for x in prim], dtype=jpeg)
                f.create_dataset('wrist_images_jpeg', data=[jpeg_encode_image(x, quality=95) for x in wrist], dtype=jpeg)
                f.create_dataset('proprio', data=np.stack(prop))
                f.create_dataset('actions', data=np.stack(acts))
                f.attrs['success'] = success
                f.attrs['task_description'] = desc
    env.close()
print(f'SUCCESS {np.mean([r["success"] for r in results]):.4f} over {len(results)} episodes', flush=True)

In [ ]:
%%writefile /content/code/report.py
import argparse
import json
import os

import numpy as np

p = argparse.ArgumentParser()
p.add_argument('--runs', nargs='*', default=[], help='rollout output folders; the first one is the baseline for paired differences')
p.add_argument('--scores', nargs='*', default=[], help='judge score files')
a = p.parse_args()


def auroc(s, y):
    s, y = np.asarray(s), np.asarray(y, bool)
    pos, neg = s[y], s[~y]
    return float(((pos[:, None] > neg[None]) + 0.5 * (pos[:, None] == neg[None])).mean()) if len(pos) and len(neg) else float('nan')


def boot_ci(x, n=5000):
    rng = np.random.default_rng(0)
    m = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)


if a.runs:
    runs = {r: {(e['task'], e['ep']): e['success'] for e in map(json.loads, open(os.path.join(r, 'episodes.jsonl')))} for r in a.runs}
    base = runs[a.runs[0]]
    print('| run | episodes | success | per task | vs first run (paired, 95% CI) |\n|---|---|---|---|---|')
    for r, d in runs.items():
        tasks = sorted({t for t, _ in d})
        per = ' '.join(f'{np.mean([v for (t, _), v in d.items() if t == k]):.2f}' for k in tasks)
        shared = [k for k in d if k in base]
        diff = np.array([d[k] - base[k] for k in shared], float)
        lo, hi = boot_ci(diff) if r != a.runs[0] and len(diff) else (0, 0)
        cmp = '-' if r == a.runs[0] else f'{diff.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]'
        print(f'| {os.path.basename(r.rstrip("/"))} | {len(d)} | {np.mean(list(d.values())):.3f} | {per} | {cmp} |')

for f in a.scores:
    rows = [json.loads(l) for l in open(f)]
    groups = {}
    for r in rows:
        groups.setdefault((r['task'], r['ep']), []).append(r)
    k = max(len(g) for g in groups.values())
    groups = {s: g for s, g in groups.items() if len(g) == k}
    single = np.mean([r['success'] for g in groups.values() for r in g])
    oracle = np.mean([any(r['success'] for r in g) for g in groups.values()])
    mixed = [g for g in groups.values() if 0 < sum(r['success'] for r in g) < len(g)]
    print(f'\n{os.path.basename(f)}: {len(groups)} start states x {k} attempts; single attempt {single:.3f}, best-of-{k} ceiling {oracle:.3f}, mixed states {len(mixed)}')
    print(f'| score | AUROC | best-of-{k} pick success | picks a success on mixed states |\n|---|---|---|---|')
    for key in ('progress', 'success_prob'):
        pick = lambda g: max(g, key=lambda r: r[key][-1])['success']
        print(f'| final {key} | {auroc([r[key][-1] for r in rows], [r["success"] for r in rows]):.3f} | '
              f'{np.mean([pick(g) for g in groups.values()]):.3f} | {sum(pick(g) for g in mixed)}/{len(mixed)} |')

## Setup

In [ ]:
!bash $CODE/setup_colab.sh 0 && cp -r $OUT/tasks/* $CODE/tasks/ 2>/dev/null; ls $CODE/tasks

## Prepare the data folder and convert the base checkpoint to the trainer's format

In [ ]:
%%bash
source /content/env.sh
SNAP=$(ls -d /root/.cache/huggingface/hub/models--nvidia--Cosmos-Policy-LIBERO-Predict2-2B/snapshots/*)
mkdir -p /content/ftdata/empty /content/ftdata/set
cp $SNAP/libero_dataset_statistics.json /content/ftdata/empty/dataset_statistics.json
echo '{}' > /content/ftdata/empty/dataset_statistics_post_norm.json
tar xf $OUT/ftset_$SET.tar -C /content/ftdata/set --strip-components=1
ls /content/ftdata/set/*/ | wc -l
[ -d /content/init_dcp/model ] || $CPV -c "from torch.distributed.checkpoint.format_utils import torch_save_to_dcp; torch_save_to_dcp('$SNAP/Cosmos-Policy-LIBERO-Predict2-2B.pt', '/content/init_dcp/model')"

## Train

In [ ]:
%%bash
source /content/env.sh
SNAP=$(ls -d /root/.cache/huggingface/hub/models--nvidia--Cosmos-Policy-LIBERO-Predict2-2B/snapshots/*)
cd /content/cosmos-policy
export IMAGINAIRE_OUTPUT_ROOT=/content/ftout
$CPV -m torch.distributed.run --nproc_per_node=1 $CODE/train_lora.py --config=cosmos_policy/config/config.py -- \
  experiment=cosmos_predict2_2b_480p_libero \
  dataloader_train.dataset.data_dir=/content/ftdata/empty \
  dataloader_train.dataset.t5_text_embeddings_path=$SNAP/libero_t5_embeddings.pkl \
  dataloader_train.dataset.rollout_data_dir=/content/ftdata/set \
  dataloader_train.dataset.demonstration_sampling_prob=1.0 \
  dataloader_train.dataset.treat_success_rollouts_as_demos=true \
  dataloader_train.batch_size=12 dataloader_train.num_workers=8 trainer.grad_accum_iter=2 \
  optimizer.lr=1e-4 "scheduler.warm_up_steps=[100,0]" "scheduler.cycle_lengths=[$ITERS,100000000000000]" \
  "scheduler.f_max=[1.0,0.3]" "scheduler.f_min=[0.3,0.3]" \
  trainer.max_iter=$ITERS trainer.logging_iter=20 checkpoint.save_iter=500 \
  checkpoint.load_path=/content/init_dcp checkpoint.load_training_state=false \
  model.config.use_lora=true model.config.lora_rank=32 model.config.lora_alpha=32 \
  job.name=ft_$SET job.wandb_mode=disabled > /content/train_$SET.log 2>&1
echo "exit $?"
cp /content/train_$SET.log $OUT/
grep -E "LORA|iter" /content/train_$SET.log | tail -n 5

## Merge the LoRA into the base checkpoint

In [ ]:
%%bash
source /content/env.sh
CK=$(find /content/ftout -type d -path "*ft_$SET/checkpoints/iter_*" | sort | tail -n 1)
echo $CK
$CPV $CODE/merge_lora.py --ckpt $CK --out /content/ft_$SET.pt --save_lora $OUT/lora_$SET.pt

## Evaluate on held-out start states 25-49

In [ ]:
%%bash
source /content/env.sh
cd /content/cosmos-policy
for r in $(seq 0 $((EVAL_SEEDS - 1))); do
  $CPV $CODE/rollout.py --out /content/runs/ft_${SET}_heldout_r$r $TASK_ARGS --episodes 25:50 --rseed $r --ckpt /content/ft_$SET.pt > /content/ft_${SET}_r$r.log 2>&1
  tail -n 1 /content/ft_${SET}_r$r.log
  mkdir -p $OUT/ft_${SET}_heldout_r$r && cp /content/runs/ft_${SET}_heldout_r$r/episodes.jsonl $OUT/ft_${SET}_heldout_r$r/
done

In [ ]:
%%bash
source /content/env.sh
for r in $(seq 0 $((EVAL_SEEDS - 1))); do
  $CPV $CODE/report.py --runs $OUT/base_heldout_r$r $OUT/ft_${SET}_heldout_r$r
done